# Hall Measurement — live notebook plotting

Runs the `HallMeasurement` procedure from `hall_measurement.py` headlessly (no Qt GUI) using pymeasure's `Results` + `Worker`, and updates a matplotlib plot as data comes in.

Re-run the "Run measurement" cell each time you want to start a new sweep.

In [5]:
%matplotlib inline
import time
from datetime import datetime

import matplotlib.pyplot as plt
from IPython import display

from pymeasure.experiment import Results, Worker
from hall_measurement import HallMeasurement

## Configure the sweep

Same parameters as the GUI, plus `settling_time` which the GUI intentionally hides.

In [6]:
procedure = HallMeasurement()

procedure.sense_current = 1e-3        # A
procedure.compliance_voltage = 2      # V
procedure.start_field = -0.5          # T
procedure.end_field = 0.5             # T
procedure.field_points = 21
procedure.field_coefficient = 0.1     # T/A
procedure.averages = 5
procedure.settling_time = 1.0         # s, not exposed in the GUI

data_filename = f"HallMeasurement_{datetime.now():%Y%m%d_%H%M%S}.csv"
data_filename

'HallMeasurement_20260729_150343.csv'

## Run measurement with live plotting

`Worker` runs `procedure` in a background thread and streams rows into `results.data` (a pandas `DataFrame`) as they're emitted. This cell polls that DataFrame and redraws the plot until the worker finishes.

Interrupt the kernel (Stop button) to abort the sweep early — the procedure's `should_stop()` check will pick it up and shut the instruments down safely.

In [3]:
results = Results(procedure, data_filename)
worker = Worker(results)
worker.start()

fig, ax = plt.subplots()

try:
    while worker.is_alive():
        worker.join(timeout=0.5)
        data = results.data

        ax.clear()
        if not data.empty:
            ax.errorbar(
                data["Magnetic Field (T)"],
                data["Hall Voltage (V)"],
                yerr=data["Hall Voltage Std (V)"],
                marker="o",
            )
        ax.set_xlabel("Magnetic Field (T)")
        ax.set_ylabel("Hall Voltage (V)")
        ax.set_title(f"status: {procedure.status}")

        display.clear_output(wait=True)
        display.display(fig)
finally:
    display.clear_output(wait=True)
    plt.close(fig)

print(f"Finished with status: {procedure.status}")
print(f"Data saved to: {data_filename}")

Finished with status: 1
Data saved to: HallMeasurement_20260729_140410.csv


## Inspect results

In [4]:
results.data

,Magnetic Field (T),Magnet Current (A),Hall Voltage (V),Hall Voltage Std (V),Hall Resistance (ohm)


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    results.data["Magnetic Field (T)"],
    results.data["Hall Voltage (V)"],
    yerr=results.data["Hall Voltage Std (V)"],
    marker="o",
)
ax.set_xlabel("Magnetic Field (T)")
ax.set_ylabel("Hall Voltage (V)")
plt.show()